# Stage 3: OCRoscope-Based OCR Filtering
This notebook runs **after** `midtraining_extraction.ipynb`. It applies a second, stronger OCR-quality filter to the extracted corpus using [Pleias/OCRoscope](https://github.com/Pleias/OCRoscope).

**Why a second pass?** Stage 2 (Internet Archive) used a lightweight dictionary-hit-ratio proxy (`ocr_score_gen`) as its OCR gate, and Stage 1 (`jbduran/think-dataset`) shipped its own precomputed `ocr_score_gen`. Neither actually inspects the *structural* OCR failure modes OCRoscope targets: garbled character runs, and long stretches whose detected language doesn't match the document. OCRoscope scores each doc with a language-model-free method (7-gram language-consistency over long segments + a non-character ratio), giving us an independent quality signal to prune the remaining garbage the Stage 2 heuristic let through.

**What OCRoscope returns** (via `ocr_evaluation(...).calculate_ocr_rate()`):
- `ratio_segment` — standardized OCR-quality rate, **0–100**, higher is cleaner. Computed as `100 - unidentified_segment_fraction*100` over long text segments.
- `ratio_nonchar` — fraction of non-character (digit/symbol) content, **0–100**, lower is more text-like.
- `probability` — mean language-detection confidence.

**Pipeline:** read the Stage 2 shards back from `zachnorton03/think-midtrain/stage2` on HF → score every doc with OCRoscope → drop docs below the quality gate → write a filtered copy to `.../stage3` → QA plots comparing the two OCR signals.

**Resumability.** Both long-running stages survive a Colab disconnect and continue instead of restarting:
- **Scoring** is checkpointed *per source shard* to `stage3/scored/`. A rerun skips any shard already scored there and only scores the missing ones. Because scored output is persisted independently of the gate, re-tuning the gate never re-scores.
- **Uploading** is idempotent and keyed on a stable, gate-independent packing order (sorted by a deterministic per-doc `uid`), so shard *N* always contains the same documents. A rerun skips shard indices already present under `stage3/data/`.

In [ ]:
# OCRoscope is NOT on PyPI -- `pip install ocroscope` 404s. Install it straight
# from the Pleias GitHub repo. Its one dependency, pycld2 (Compact Language
# Detector 2), is a C extension: on modern Colab the prebuilt wheel usually
# installs fine, but if it has to build from source it needs libicu headers,
# which the apt line provides. We install pycld2 first so a build failure is
# obvious here rather than buried in the git install.
!apt-get -qq install -y libicu-dev > /dev/null
!pip install -q pycld2
!pip install -q "git+https://github.com/Pleias/OCRoscope.git"
!pip install -q pyarrow tqdm huggingface_hub tiktoken

# Sanity check that the package imported from GitHub is the real one.
import ocroscope
print("ocroscope installed from:", ocroscope.__file__)

In [ ]:
from ocroscope import ocr_evaluation

# Smoke-test the API on a trivially clean and a trivially garbled string so we
# know the import + pycld2 backend work before spending time streaming shards.
clean = ("The mathematical theory of probability owes its origin to a "
         "question proposed by a gambler to Blaise Pascal in the year 1654. "
         "From this modest beginning the subject has grown into a vast and "
         "rigorous branch of modern analysis.") * 6
garbled = "th3 m4th3m t1c l th30ry rn f pr b b l ty 0w s 1ts 0r g n t0 qu st10n" * 40

for label, txt in [("clean", clean), ("garbled", garbled)]:
    ev = ocr_evaluation(id=label, text=txt)
    ev.calculate_ocr_rate()
    print(f"{label:8s}  ratio_segment={ev.ratio_segment:6.2f}  "
          f"ratio_nonchar={ev.ratio_nonchar:6.2f}  probability={ev.probability:6.2f}")

## 1. Configuration & HF connection
We read the Stage 2 output written by `midtraining_extraction.ipynb` and write a filtered Stage 3 copy alongside it. The HF token comes from Colab Secrets (`HF_TOKEN_Z`), exactly as in the Stage 2 notebook.

In [ ]:
import os, json, io
# Cap the per-chunk read timeout for HF downloads BEFORE importing huggingface_hub,
# so a CDN node that hangs mid-stream (the "reconstructing file" bar freezing at
# e.g. 87%) raises a ReadTimeout instead of blocking forever. robust_hf_download
# (defined below) then retries on that timeout and re-resolves to a healthy node.
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '30')  # seconds per chunk read

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import tqdm
from huggingface_hub import HfApi, hf_hub_download, list_repo_files
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN_Z')
HF_REPO_ID = 'zachnorton03/think-midtrain'
SRC_PREFIX = 'stage2'          # input: OCR-heuristic-filtered IA corpus
DST_PREFIX = 'stage3'          # output: OCRoscope-filtered corpus
SCORED_PREFIX = f'{DST_PREFIX}/scored'   # per-source-shard scoring checkpoints
DATA_PREFIX = f'{DST_PREFIX}/data'       # final filtered/packed shards
CHARS_PER_SHARD = 250_000_000  # keep the same shard size as upstream stages

api = HfApi(token=HF_TOKEN)

# --- OCRoscope quality gate --------------------------------------------------
# ratio_segment is 0-100, higher = cleaner. 85 is a deliberately conservative
# floor: on the smoke test above a clean passage scores ~95-100 and obvious
# garble scores far lower. Tune after inspecting the score distribution in the
# QA section below (the histogram there makes a good gate obvious).
MIN_RATIO_SEGMENT = 85.0
# ratio_nonchar is 0-100, higher = more digits/symbols (tables, indices, noise).
# Pre-1930 STEM legitimately contains equations and tables, so keep this loose;
# it mainly catches pages that are almost entirely numeric/symbol soup.
MAX_RATIO_NONCHAR = 40.0
# calculate_ocr_rate aborts on texts <= this many words; our MIN_CHARS=2000
# upstream gate makes this a non-issue, but guard anyway.
OCR_MIN_WORDS = 7

def list_repo_paths():
    """Current set of files in the dataset repo. Re-listed (not cached) each call
    so the resume checks see checkpoints written earlier in THIS run too."""
    return set(list_repo_files(repo_id=HF_REPO_ID, repo_type='dataset', token=HF_TOKEN))

def hf_file_exists(path_in_repo, _listing=None):
    """True if path_in_repo already exists in the dataset repo. Used by the
    resume logic so a rerun can skip work that already landed on HF. Pass a
    precomputed listing to avoid re-listing the repo per call."""
    return path_in_repo in (_listing if _listing is not None else list_repo_paths())

print(f"Source: {HF_REPO_ID}/{SRC_PREFIX}")
print(f"Dest:   {HF_REPO_ID}/{DST_PREFIX}")
print(f"Gate:   ratio_segment >= {MIN_RATIO_SEGMENT}, ratio_nonchar <= {MAX_RATIO_NONCHAR}")

In [ ]:
# Enumerate the Stage 2 shards on HF. midtraining_extraction.ipynb wrote them to
# stage2/data/shard_XXXXX.parquet -- but IMPORTANT: those parquet files contain
# ONLY a 'text' column. All metadata (title_src, ocr_score_gen, char_count,
# token counts, topic, year, ia_identifier, ...) lives SEPARATELY in
# stage2/audit_metadata.jsonl, aligned row-for-row: within each shard the Nth
# text row corresponds to the Nth audit record whose output_shard_filename is
# that shard (same append order flush_shard used). We load that metadata here and
# join it back on during scoring.
src_shards = sorted(
    f for f in list_repo_files(repo_id=HF_REPO_ID, repo_type='dataset', token=HF_TOKEN)
    if f.startswith(f'{SRC_PREFIX}/data/') and f.endswith('.parquet')
)
print(f"Found {len(src_shards)} Stage 2 shard(s).")
assert src_shards, "No Stage 2 shards found -- run midtraining_extraction.ipynb first."

# Load the audit metadata (small: no text) and index it by shard filename, in the
# original per-shard append order so positional alignment to the text rows holds.
audit_local = hf_hub_download(repo_id=HF_REPO_ID, filename=f'{SRC_PREFIX}/audit_metadata.jsonl',
                              repo_type='dataset', token=HF_TOKEN, etag_timeout=30)
audit_records = []
with open(audit_local) as f:
    for line in f:
        audit_records.append(json.loads(line))
df_audit = pd.DataFrame(audit_records)

# Scalarize list/dict-valued cells. Internet Archive metadata fields (title,
# subject, collection, creator, ...) are frequently returned as LISTS, so
# columns like title_src end up mixed str/list. PyArrow can't write a mixed
# object column (ArrowTypeError: "Expected bytes, got a 'list' object"), so we
# flatten every non-scalar cell to a string once, here, before anything joins to
# it -- fixes the packing write and keeps the QA display clean too.
def _scalarize(v):
    if isinstance(v, (list, tuple)):
        return '; '.join(str(x) for x in v)
    if isinstance(v, dict):
        return json.dumps(v, ensure_ascii=False)
    return v

_obj_cols = [c for c in df_audit.columns if df_audit[c].dtype == object]
for c in _obj_cols:
    df_audit[c] = df_audit[c].map(_scalarize)
print(f"Loaded {len(df_audit):,} audit metadata records; scalarized "
      f"{len(_obj_cols)} object column(s).")
print(f"Columns: {sorted(df_audit.columns)[:12]}{' ...' if len(df_audit.columns) > 12 else ''}")

# Sanity: audit metadata must reference the shards by filename so we can join.
assert 'output_shard_filename' in df_audit.columns, \
    "audit_metadata.jsonl lacks output_shard_filename -- cannot align to text shards."

# Group audit rows by shard, preserving order -> the positional key for the join.
audit_by_shard = {name: grp.reset_index(drop=True)
                  for name, grp in df_audit.groupby('output_shard_filename', sort=False)}
print(f"Audit metadata covers {len(audit_by_shard)} distinct shard filenames.")

## 2. Score every document with OCRoscope (resumable, per-shard checkpoints)
Each Stage 2 shard is scored independently and its scored copy is written to `stage3/scored/<shard>.parquet` **before** moving to the next shard. On a rerun, any source shard whose scored file already exists is skipped and simply re-read — so a disconnect costs at most the one shard that was in flight. Scoring is deliberately separated from the gate: the persisted scores don't encode any threshold, so you can re-tune `MIN_RATIO_SEGMENT` / `MAX_RATIO_NONCHAR` later without re-scoring anything.

Each doc also gets a deterministic `uid` (`<src_shard_stem>:<row>`) that the upload stage uses for a stable, gate-independent packing order.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time, random

def score_text(text):
    """Run OCRoscope on one document. Returns (ratio_segment, ratio_nonchar,
    probability, ok). `ok` is False when the text is too short for a verdict --
    those are treated as failures so we never keep an unscoreable doc."""
    if not isinstance(text, str) or len(text.split()) <= OCR_MIN_WORDS:
        return (0.0, 100.0, 0.0, False)
    try:
        ev = ocr_evaluation(id=None, text=text)
        ev.calculate_ocr_rate(ocr_detection_segment=OCR_MIN_WORDS)
        # calculate_ocr_rate prints + returns early (leaving attrs unset) when
        # the text is too short; guard against that.
        rs = getattr(ev, 'ratio_segment', None)
        rn = getattr(ev, 'ratio_nonchar', None)
        pr = getattr(ev, 'probability', 0.0)
        if rs is None or rn is None:
            return (0.0, 100.0, 0.0, False)
        return (float(rs), float(rn), float(pr), True)
    except Exception:
        # A scoring crash shouldn't take down the whole run; treat as a fail.
        return (0.0, 100.0, 0.0, False)

SCORE_WORKERS = 8

def score_dataframe(df):
    """Adds ocroscope_* columns to df in place (row order preserved)."""
    texts = df['text'].tolist()
    with ThreadPoolExecutor(max_workers=SCORE_WORKERS) as ex:
        results = list(ex.map(score_text, texts))
    df['ocroscope_ratio_segment'] = [r[0] for r in results]
    df['ocroscope_ratio_nonchar'] = [r[1] for r in results]
    df['ocroscope_probability']   = [r[2] for r in results]
    df['ocroscope_scored']        = [r[3] for r in results]
    return df

def scored_path_for(src_shard_path):
    """HF path of the scored checkpoint for a given stage2 source shard."""
    stem = os.path.splitext(os.path.basename(src_shard_path))[0]
    return f"{SCORED_PREFIX}/{stem}.parquet"

# --- Self-healing download -------------------------------------------------
# hf_hub_download has NO wall-clock timeout on the body transfer: if an HF CDN
# node hangs mid-stream (seen as a "reconstructing file" progress bar frozen at
# e.g. 87%), the call blocks forever and the whole cell stalls. This wrapper
# bounds each attempt and retries on any failure -- a fresh attempt re-resolves
# to (usually) a healthy node, exactly like Stage 2's IA fetcher did. We force a
# clean re-download each retry (force_download=True) so a half-written cache
# entry from a killed transfer can't poison the retry.
DL_MAX_RETRIES = 5
DL_ETAG_TIMEOUT = 30          # seconds to resolve metadata/etag before giving up
DL_ATTEMPT_TIMEOUT = 600      # hard ceiling per attempt (a 150MB shard at ~10MB/s ~ 15s)

def robust_hf_download(filename, force_on_retry=True):
    """hf_hub_download with bounded retries + backoff. Raises only if every
    attempt fails, so a persistent outage still surfaces instead of hanging."""
    last_err = None
    for attempt in range(DL_MAX_RETRIES):
        try:
            return hf_hub_download(
                repo_id=HF_REPO_ID, filename=filename, repo_type='dataset',
                token=HF_TOKEN, etag_timeout=DL_ETAG_TIMEOUT,
                force_download=(force_on_retry and attempt > 0),
            )
        except Exception as e:
            last_err = e
            wait = min(60, (2 ** attempt) + random.random())
            tqdm.tqdm.write(f"  download retry {attempt+1}/{DL_MAX_RETRIES} for "
                            f"{filename} after error: {type(e).__name__}: {e} "
                            f"(sleeping {wait:.1f}s)")
            time.sleep(wait)
    raise RuntimeError(f"Failed to download {filename} after {DL_MAX_RETRIES} attempts") from last_err

In [ ]:
# --- Resumable scoring loop (memory-frugal, metadata-joined) ---------------
# For each source shard: if its scored checkpoint exists on HF, load only the
# scores from it (no re-scoring); otherwise download the text shard, score it,
# and upload a small scores-only checkpoint before moving on.
#
# METADATA JOIN: the stage2 text shards + our checkpoints hold ONLY text +
# ocroscope_* scores. Every metadata column (title_src, ocr_score_gen,
# char_count, token counts, ia_identifier, ...) comes from df_audit, joined
# POSITIONALLY per shard (Nth row <-> Nth audit record for that shard). This is
# the single source of metadata truth, so it works identically for freshly
# scored shards and for checkpoints written by an earlier notebook version.
#
# MEMORY: 'text' is the RAM hog. We keep only lightweight columns in memory and
# drop text after scoring; the packing stage re-reads text per shard on demand.
import gc
import tempfile
_score_tmp = tempfile.mkdtemp()

# Scores the checkpoint stores (besides text). Metadata is NOT stored -- it is
# rejoined from df_audit every run.
SCORE_COLS = ['ocroscope_ratio_segment', 'ocroscope_ratio_nonchar',
              'ocroscope_probability', 'ocroscope_scored']
# Lightweight metadata columns kept in memory for the gate + packing + QA.
META_LIGHT = ['title_src', 'char_count', 'token_count_o200k_base_gen',
              'ocr_score_gen', 'ia_identifier']

def audit_for(stem_shard_name):
    """Audit rows for a shard filename (e.g. 'shard_00003.parquet'), in order."""
    return audit_by_shard.get(stem_shard_name)

def build_light(stem, n_rows, score_df):
    """Assemble the in-memory light frame for one shard: positional metadata
    join + scores + deterministic uid. Asserts the row counts line up."""
    a = audit_for(stem)
    if a is None:
        raise KeyError(f"No audit metadata for {stem} -- cannot align.")
    assert len(a) == n_rows, (
        f"Row-count mismatch for {stem}: {n_rows} text rows vs {len(a)} audit rows. "
        f"Positional join is unsafe; do NOT proceed.")
    light = pd.DataFrame(index=range(n_rows))
    for c in META_LIGHT:
        light[c] = a[c].values if c in a.columns else np.nan
    for c in SCORE_COLS:
        light[c] = score_df[c].values
    light['src_shard'] = f'{SRC_PREFIX}/data/{stem}'
    light['uid'] = [f'{os.path.splitext(stem)[0]}:{i}' for i in range(n_rows)]
    return light

_existing = list_repo_paths()

light_frames = []
for shard_path in tqdm.tqdm(src_shards, desc="Scoring shards (resumable)"):
    stem = os.path.basename(shard_path)                 # e.g. 'shard_00003.parquet'
    ckpt = scored_path_for(shard_path)

    if hf_file_exists(ckpt, _listing=_existing):
        local = robust_hf_download(ckpt)
        avail = pq.read_schema(local).names
        cols = [c for c in SCORE_COLS if c in avail]
        score_df = pq.read_table(local, columns=cols).to_pandas()
        light = build_light(stem, len(score_df), score_df)
        tqdm.tqdm.write(f"  resume: {ckpt} ({len(light):,} docs, already scored)")
    else:
        local = robust_hf_download(shard_path)
        df = pq.read_table(local).to_pandas()           # {'text': ...}
        score_dataframe(df)                             # adds ocroscope_* columns
        # Persist a SMALL scores-only checkpoint (text + scores) BEFORE continuing
        # -> the resumable boundary and the text source for packing. No metadata
        #    stored: it's always rejoined from df_audit.
        ckpt_df = df[['text'] + SCORE_COLS]
        out_local = os.path.join(_score_tmp, stem)
        pq.write_table(pa.Table.from_pandas(ckpt_df, preserve_index=False), out_local)
        api.upload_file(
            path_or_fileobj=out_local, path_in_repo=ckpt,
            repo_id=HF_REPO_ID, repo_type='dataset',
            commit_message=f'Stage 3 scored checkpoint {stem}',
        )
        os.remove(out_local)
        light = build_light(stem, len(df), df)
        del df, ckpt_df
        tqdm.tqdm.write(f"  scored + saved: {ckpt} ({len(light):,} docs)")

    light_frames.append(light)
    gc.collect()

df_scored = pd.concat(light_frames, ignore_index=True)
del light_frames
gc.collect()
assert df_scored['uid'].is_unique, "uid collision -- packing determinism broken"
print(f"\nScored corpus: {len(df_scored):,} documents across {len(src_shards)} shard(s). "
      f"(text NOT held in memory -- re-read per shard during packing)")
display(df_scored[['uid', 'title_src', 'ocr_score_gen',
                   'ocroscope_ratio_segment', 'ocroscope_ratio_nonchar',
                   'ocroscope_probability']].head(10))

## 3. Inspect the score distribution before committing to a gate
The histograms below let you pick `MIN_RATIO_SEGMENT` / `MAX_RATIO_NONCHAR` from the data rather than guessing. The scatter compares OCRoscope's `ratio_segment` against Stage 2's dictionary-hit `ocr_score_gen` — points low on the OCRoscope axis but high on the old axis are exactly the docs the first heuristic missed.

Re-tuning the gate here and rerunning downstream is cheap: scoring is already checkpointed, so nothing gets re-scored.

In [ ]:
import matplotlib.pyplot as plt

scoreable = df_scored[df_scored['ocroscope_scored']]
print(f"Scoreable docs: {len(scoreable):,} / {len(df_scored):,}")
print(scoreable[['ocroscope_ratio_segment', 'ocroscope_ratio_nonchar',
                 'ocroscope_probability']].describe())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(scoreable['ocroscope_ratio_segment'], bins=40, color='seagreen', edgecolor='black')
axes[0].axvline(MIN_RATIO_SEGMENT, color='red', linestyle='--', label=f'gate {MIN_RATIO_SEGMENT}')
axes[0].set_title('OCRoscope ratio_segment (higher = cleaner)')
axes[0].set_xlabel('ratio_segment'); axes[0].legend()

axes[1].hist(scoreable['ocroscope_ratio_nonchar'], bins=40, color='indianred', edgecolor='black')
axes[1].axvline(MAX_RATIO_NONCHAR, color='red', linestyle='--', label=f'gate {MAX_RATIO_NONCHAR}')
axes[1].set_title('OCRoscope ratio_nonchar (lower = more text)')
axes[1].set_xlabel('ratio_nonchar'); axes[1].legend()

if 'ocr_score_gen' in scoreable.columns:
    axes[2].scatter(scoreable['ocr_score_gen'], scoreable['ocroscope_ratio_segment'],
                    s=6, alpha=0.3)
    axes[2].axhline(MIN_RATIO_SEGMENT, color='red', linestyle='--')
    axes[2].set_title('Stage 2 dict-hit vs. OCRoscope')
    axes[2].set_xlabel('ocr_score_gen (Stage 2)'); axes[2].set_ylabel('ocroscope_ratio_segment')
else:
    axes[2].set_visible(False)

plt.tight_layout(); plt.show()

## 4. Apply the gate and report what gets pruned

In [ ]:
passed = (
    df_scored['ocroscope_scored'] &
    (df_scored['ocroscope_ratio_segment'] >= MIN_RATIO_SEGMENT) &
    (df_scored['ocroscope_ratio_nonchar'] <= MAX_RATIO_NONCHAR)
)
# Sort by uid so packing order is deterministic and gate-independent -- this is
# what makes the upload stage's shard-skip resume logic sound.
df_pass = df_scored[passed].sort_values('uid').reset_index(drop=True)
df_fail = df_scored[~passed]

# df_scored carries no 'text' column (dropped for memory), so char/token totals
# come from the count columns, which are always present from Stage 2 metadata.
def _chars(d):  return int(d['char_count'].sum())
def _tokens(d): return int(d['token_count_o200k_base_gen'].sum()) if 'token_count_o200k_base_gen' in d else None

print("--- OCRoscope gate results ---")
print(f"Input docs:   {len(df_scored):,}")
print(f"Passed:       {len(df_pass):,}  ({100*len(df_pass)/max(1,len(df_scored)):.1f}%)")
print(f"Dropped:      {len(df_fail):,}")
print(f"Chars kept:   {_chars(df_pass):,}  (dropped {_chars(df_fail):,})")
tk = _tokens(df_pass)
if tk is not None:
    print(f"Tokens kept:  {tk:,}")

print("\nSample of DROPPED docs (lowest ratio_segment) -- eyeball these to sanity-check the gate:")
cols = [c for c in ['title_src','ocr_score_gen','ocroscope_ratio_segment','ocroscope_ratio_nonchar'] if c in df_fail.columns]
display(df_fail.sort_values('ocroscope_ratio_segment').head(10)[cols])

In [ ]:
# Optional: eyeball the actual text of a few dropped docs to confirm they are
# genuinely garbled and not false positives (e.g. heavily equation-laden math).
# df_scored holds no text, and checkpoints hold text positionally (no uid column),
# so we fetch text ON DEMAND: read the source checkpoint's text column and index
# by the row number encoded in the uid.
def fetch_texts_for(rows):
    """rows: DataFrame slice with 'uid' and 'src_shard'. Returns {uid: text}."""
    out = {}
    for src_shard, grp in rows.groupby('src_shard'):
        ckpt = scored_path_for(src_shard)
        local = robust_hf_download(ckpt)
        text_col = pq.read_table(local, columns=['text']).column('text').to_pylist()
        for _, r in grp.iterrows():
            out[r['uid']] = text_col[int(r['uid'].rsplit(':', 1)[1])]
    return out

worst = df_fail.sort_values('ocroscope_ratio_segment').head(3)
texts = fetch_texts_for(worst)
for _, row in worst.iterrows():
    print("=" * 100)
    print(f"{row.get('title_src','?')}  |  ratio_segment={row['ocroscope_ratio_segment']:.1f}  "
          f"ratio_nonchar={row['ocroscope_ratio_nonchar']:.1f}")
    print("-" * 100)
    print((texts.get(row['uid']) or '')[:800])
    print("...")

## 5. Re-pack the survivors into Stage 3 shards and upload (resumable)
Same ~250M-char packing as upstream. Packing is computed from the `uid`-sorted survivor list, so the *contents* of shard *N* are fully determined by the corpus + gate — not by run order. The upload loop **skips any shard index already present** under `stage3/data/`, so a rerun after a disconnect resumes at the first missing shard instead of re-uploading or duplicating.

> If you change the gate after some Stage 3 shards were already uploaded, the shard boundaries shift and old shards no longer match. In that case run the **"reset Stage 3 outputs"** cell below first, then re-pack from scratch. Leaving the gate fixed keeps reruns purely additive.

In [ ]:
# --- OPTIONAL: reset Stage 3 packed shards -----------------------------------
# Only needed if you CHANGED THE GATE after already uploading some stage3/data/
# shards (shard boundaries shift, so a partial old set would be inconsistent).
# This clears stage3/data/ AND the manifest, but deliberately KEEPS stage3/scored/
# so you don't have to re-score. Set to True to actually delete.
CONFIRM_RESET_PACKED = False

if CONFIRM_RESET_PACKED:
    stale = [
        f for f in list_repo_files(repo_id=HF_REPO_ID, repo_type='dataset', token=HF_TOKEN)
        if f.startswith(f'{DATA_PREFIX}/') or f == f'{DST_PREFIX}/manifest.json'
    ]
    print(f"Deleting {len(stale)} packed file(s) under {DATA_PREFIX}/ (keeping scored/) ...")
    for f in stale:
        api.delete_file(path_in_repo=f, repo_id=HF_REPO_ID, repo_type='dataset',
                        commit_message=f'Reset Stage 3 packed: remove {f}')
        print(f"  deleted {f}")
    print("Reset complete. Re-run the packing cell below to repack under the new gate.")
else:
    print("Reset skipped (CONFIRM_RESET_PACKED=False). Packing cell will resume additively.")

In [ ]:
# --- Memory-frugal, resumable packing --------------------------------------
# df_pass has metadata + scores but NO text. The checkpoints have text + scores
# but NO metadata. So each output row = metadata/scores from df_pass + text
# streamed from the source checkpoint, joined POSITIONALLY (uid encodes the row
# index within its source shard). We stream ONE checkpoint at a time and pack
# survivors into ~CHARS_PER_SHARD output shards, so peak memory is ~one source
# shard's text + one output shard, never the whole corpus.
#
# Determinism/resume: survivors are packed in uid order, so output shard N always
# holds the same docs. Already-uploaded output shard indices are skipped; missing
# ones are rebuilt from the deterministic plan.
import gc

# Full metadata to write per doc: everything df_audit provides + our scores.
# (We reload full audit rows per shard rather than only META_LIGHT so the output
#  keeps the complete Stage 2 schema.)
SCORE_OUT = ['ocroscope_ratio_segment', 'ocroscope_ratio_nonchar',
             'ocroscope_probability', 'ocroscope_scored']

# 1. Deterministic plan from the light table (char counts only, no text).
pass_sorted = df_pass.sort_values('uid').reset_index(drop=True)
plan = []                      # plan[k] = list of uids for output shard k
cur, cur_chars = [], 0
for uid, ch in zip(pass_sorted['uid'], pass_sorted['char_count']):
    cur.append(uid); cur_chars += int(ch)
    if cur_chars >= CHARS_PER_SHARD:
        plan.append(cur); cur, cur_chars = [], 0
if cur:
    plan.append(cur)
print(f"Planned {len(plan)} output shard(s) from {len(pass_sorted):,} survivors.")

uid_to_out = {u: k for k, uids in enumerate(plan) for u in uids}
# uid -> its metadata+scores row (as a dict) from df_pass, for the output join.
uid_meta = {r['uid']: r for r in pass_sorted.to_dict('records')}
# uid -> row index within its source shard (the positional text key).
def uid_row_index(u):
    return int(u.rsplit(':', 1)[1])

existing_data = {
    os.path.basename(f)
    for f in list_repo_files(repo_id=HF_REPO_ID, repo_type='dataset', token=HF_TOKEN)
    if f.startswith(f'{DATA_PREFIX}/') and f.endswith('.parquet')
}
missing_out_idx = {k for k in range(len(plan))
                   if f'shard_{k:05d}.parquet' not in existing_data}
uploaded = [f'shard_{k:05d}.parquet' for k in range(len(plan))]
if existing_data:
    print(f"Resume: {len(existing_data)} output shard(s) already uploaded -- skipping those.")

from collections import defaultdict
buffers = defaultdict(list)    # out_idx -> list of output row dicts (meta + text)
pack_tmp = tempfile.mkdtemp()

def _write_output_shard(k):
    shard_name = f'shard_{k:05d}.parquet'
    rows = buffers.pop(k, [])
    rows.sort(key=lambda r: r['uid'])
    out_df = pd.DataFrame(rows).drop(columns=['uid', 'src_shard'], errors='ignore')
    out_df['output_shard_filename'] = shard_name
    out_df['output_shard_index'] = k
    local_path = os.path.join(pack_tmp, shard_name)
    pq.write_table(pa.Table.from_pandas(out_df, preserve_index=False), local_path)
    api.upload_file(
        path_or_fileobj=local_path, path_in_repo=f'{DATA_PREFIX}/{shard_name}',
        repo_id=HF_REPO_ID, repo_type='dataset',
        commit_message=f'Stage 3 (OCRoscope-filtered) {shard_name}',
    )
    os.remove(local_path)
    del out_df, rows
    gc.collect()
    tqdm.tqdm.write(f"  wrote {shard_name}")

# Survivor uids grouped by source shard so each checkpoint is opened once.
pass_by_src = pass_sorted.groupby('src_shard')['uid'].apply(list).to_dict()

for src_shard in tqdm.tqdm(sorted(pass_by_src), desc="Packing (streaming text)"):
    uids = pass_by_src[src_shard]
    if not ({uid_to_out[u] for u in uids} & missing_out_idx):
        continue  # every output shard this source feeds is already uploaded

    ckpt = scored_path_for(src_shard)
    local = robust_hf_download(ckpt)
    text_col = pq.read_table(local, columns=['text']).column('text').to_pylist()
    for u in uids:
        k = uid_to_out[u]
        if k not in missing_out_idx:
            continue
        row = dict(uid_meta[u])            # metadata + scores (+ uid, src_shard)
        row['text'] = text_col[uid_row_index(u)]
        buffers[k].append(row)
    del text_col
    gc.collect()

    for k in list(buffers.keys()):
        if len(buffers[k]) == len(plan[k]):
            _write_output_shard(k)

for k in sorted(list(buffers.keys())):
    _write_output_shard(k)

print(f"Stage 3 packed set: {len(uploaded)} shard(s) total in {HF_REPO_ID}/{DATA_PREFIX}/")

In [ ]:
# Manifest: record exactly which gate produced this Stage 3 corpus.
# Written last, after all shards are confirmed present, so its existence is a
# signal that the packing stage finished cleanly.
manifest = {
    'stage': 3,
    'description': 'OCRoscope-filtered copy of stage2 (Pleias/OCRoscope)',
    'source_prefix': SRC_PREFIX,
    'source_shards': len(src_shards),
    'gate': {
        'min_ratio_segment': MIN_RATIO_SEGMENT,
        'max_ratio_nonchar': MAX_RATIO_NONCHAR,
    },
    'input_docs': int(len(df_scored)),
    'output_docs': int(len(df_pass)),
    'output_shards': len(uploaded),
    'output_chars': _chars(df_pass),
    'output_tokens_o200k_base': _tokens(df_pass),
}
manifest_local = os.path.join(pack_tmp, 'manifest.json')
with open(manifest_local, 'w') as f:
    json.dump(manifest, f, indent=2)
api.upload_file(
    path_or_fileobj=manifest_local,
    path_in_repo=f'{DST_PREFIX}/manifest.json',
    repo_id=HF_REPO_ID, repo_type='dataset',
    commit_message='Stage 3 manifest',
)
print(json.dumps(manifest, indent=2))

## 6. Read-back verification
Confirm the uploaded Stage 3 corpus is intact and that the OCR gate actually held — the minimum `ocroscope_ratio_segment` in the read-back should be at or above `MIN_RATIO_SEGMENT`.

In [ ]:
check_manifest = robust_hf_download(f'{DST_PREFIX}/manifest.json')
with open(check_manifest) as f:
    print(json.dumps(json.load(f), indent=2))

first = robust_hf_download(f'{DATA_PREFIX}/{uploaded[0]}')
check_df = pq.read_table(first).to_pandas()
print(f"\nFirst Stage 3 shard: {len(check_df):,} docs")
print(f"min ocroscope_ratio_segment in shard: {check_df['ocroscope_ratio_segment'].min():.2f} "
      f"(gate was {MIN_RATIO_SEGMENT})")
assert check_df['ocroscope_ratio_segment'].min() >= MIN_RATIO_SEGMENT, "Gate leak!"
display(check_df[['title_src', 'ocroscope_ratio_segment', 'ocroscope_ratio_nonchar']].head())
print("\nStage 3 OCRoscope filtering complete.")